# Respiratory Virus Dashboard
Weekly PCR test positivity for influenza and RSV in England, 2025.

In [23]:
from pathlib import Path
import json

data_dir = Path("data")

with (data_dir / "influenza_2025.json").open(encoding="utf-8") as file:
    flu_data = json.load(file)

with (data_dir / "rsv_2025.json").open(encoding="utf-8") as file:
    rsv_data = json.load(file)

jsondata = {
    "Influenza": flu_data,
    "RSV": rsv_data,
}

In [24]:
import pandas as pd
from IPython.display import display

def wrangle_data(rawdata):
    columns = {}

    for virus, records in rawdata.items():
        table = pd.DataFrame(records)
        table["date"] = pd.to_datetime(table["date"])
        table = table.set_index("date")

        if not table.index.is_unique:
            raise ValueError("Duplicate dates found for " + virus)

        columns[virus] = pd.to_numeric(table["metric_value"])

    return pd.DataFrame(columns).sort_index()

df = wrangle_data(jsondata)

## Refresh 2025 data

In [25]:
import time
import requests

def access_api():
    metrics = {
        "Influenza": "influenza_testing_positivityByWeek",
        "RSV": "RSV_testing_positivityByWeek",
    }

    params = {
        "year": 2025,
        "age": "all",
        "sex": "all",
        "stratum": "default",
        "page_size": 365,
        "format": "json",
    }

    new_data = {}

    for virus, metric in metrics.items():
        url = (
            "https://api.ukhsa-dashboard.data.gov.uk/"
            "themes/infectious_disease/sub_themes/respiratory/"
            f"topics/{virus}/geography_types/Nation/geographies/England/"
            f"metrics/{metric}"
        )

        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        page = response.json()

        records = page["results"]

        if not records or len(records) != page["count"]:
            raise ValueError("Missing or incomplete data for " + virus)

        new_data[virus] = records
        time.sleep(0.4)

    return new_data

In [26]:
import ipywidgets as wdg

refresh_status = wdg.Label(value="2025 data loaded.")

def refresh_graph():
    graph.clear_output(wait=True)
    with graph:
        plot_viruses(virus_select.value)

def api_button_callback(button):
    global df, jsondata

    button.disabled = True
    button.description = "Refreshing..."
    button.icon = "refresh"
    refresh_status.value = "Downloading 2025 data..."

    try:
        new_data = access_api()
        new_df = wrangle_data(new_data)

    except (requests.RequestException, ValueError, KeyError, TypeError):
        refresh_status.value = "Refresh failed. Previous data retained."
        button.icon = "exclamation-triangle"

    else:
        jsondata = new_data
        df = new_df
        refresh_graph()

        refresh_status.value = f"Refreshed: {len(df)} weekly dates for 2025."
        button.icon = "check"

    finally:
        button.disabled = False
        button.description = "Refresh 2025 data"

apibutton = wdg.Button(
    description="Refresh 2025 data",
    button_style="info",
    icon="refresh",
    layout=wdg.Layout(width="200px"),
)

apibutton.on_click(api_button_callback)

display(apibutton, refresh_status)

Button(button_style='info', description='Refresh 2025 data', icon='refresh', layout=Layout(width='200px'), sty…

Label(value='2025 data loaded.')

## Graphs and Analysis

Compare weekly PCR test positivity for influenza and RSV in England during 2025.
Use the dropdown to display both viruses or either one individually.

Test positivity is the percentage of PCR tests with positive results.
It is a surveillance indicator of virus activity among those tested.

In [27]:
from math import ceil
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator, PercentFormatter
import ipywidgets as wdg

%matplotlib inline

def plot_viruses(virus):
    selected = ["Influenza", "RSV"] if virus == "Both" else [virus]
    data = df[selected]
    colors = {"Influenza": "#2563EB", "RSV": "#EA580C"}

    highest = data.max().max()
    if pd.isna(highest):
        print("No data available for this selection.")
        return

    # Fit the selected data, rounding up in 5-percentage-point steps.
    ymax = max(5, ceil(highest * 1.1 / 5) * 5)

    fig, ax = plt.subplots(figsize=(9.5, 4.5), dpi=110)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("#F8FAFC")

    for name in selected:
        ax.plot(
            data.index, data[name],
            color=colors[name], linewidth=2.8, label=name,
        )

    # Subtle shading when viewing one virus.
    if len(selected) == 1:
        name = selected[0]
        ax.fill_between(
            data.index, data[name].to_numpy(dtype=float),
            color=colors[name], alpha=0.10,
        )

    label = "Influenza and RSV" if virus == "Both" else virus
    ax.set_title(
        f"{label}: weekly PCR test positivity",
        loc="left", fontsize=15, fontweight="bold",
        color="#0F172A", pad=18,
    )

    ax.set_xlabel("Month · 2025", labelpad=10)
    ax.set_ylabel("PCR test positivity", labelpad=10)
    ax.set_ylim(0, ymax)
    ax.set_xlim(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-31"))

    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax.yaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))

    ax.grid(axis="y", color="#E2E8F0", linewidth=0.8)
    ax.set_axisbelow(True)

    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color("#CBD5E1")

    ax.tick_params(colors="#475569", labelsize=10, length=0, pad=8)
    ax.legend(frameon=False, loc="upper left", ncol=len(selected))

    fig.tight_layout()
    plt.show()

virus_select = wdg.Dropdown(
    options=["Both", "Influenza", "RSV"],
    value="Both",
    description="Show:",
    layout=wdg.Layout(width="280px"),
)

graph = wdg.interactive_output(plot_viruses, {"virus": virus_select})
display(virus_select, graph)

Dropdown(description='Show:', layout=Layout(width='280px'), options=('Both', 'Influenza', 'RSV'), value='Both'…

Output()

## Author and License

Author: Jarrod Stringer

Based on UK Government [data](https://ukhsa-dashboard.data.gov.uk/)
published by the [UK Health Security Agency](https://www.gov.uk/government/organisations/uk-health-security-agency)
and on the [DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash)
by Fabrizio Smeraldi.

Released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/gpl-3.0.html).